In [1]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from skimage.measure import find_contours

from unc_handling import UG_prompter
from DataLoader import DataLoader
from segmentation import Segmentation
from segmentation_util import combine_prompt_sets
from evaluation import Evaluator, compare_to_recontours

root = r"C:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\data\LUNDPROBE\ExtendedSamples\development"
methods_available = ["raycast", "local_normals"]
method = methods_available[1]
run_eval = True
evaluate_compare_recontours = True

data = DataLoader(parentfolder=root,subject_nr=0,volume_of_interest="CTVT",verbose=True)
unc_handler = UG_prompter(data=data)
seg_handler = Segmentation(data=data)

unc_handler.threshold_uncertainty_map(unc_threshold=None, target_mm=3.0, method="raycast", mode="median") #unc_threshold=0.033470
unc_handler.compute_band_thickness(method=method)

Loaded subject newAcq_050f229dc2bdb64c with volume of interest 'CTVT'
Image shape: (88, 1024, 1024), Mask shape: (88, 1024, 1024), Uncertainty map shape: (88, 1024, 1024), Ground truth shape: (88, 1024, 1024)
Image spacing (z, y, x): (2.5, 0.46880000829696655, 0.46880000829696655)
Initilialized UG_prompter for subject 0 with volume of interest 'CTVT'
Image shape: (88, 1024, 1024), Mask shape: (88, 1024, 1024), Uncertainty map shape: (88, 1024, 1024).
Initialized Segmentation for subject 0 with volume of interest 'CTVT'
Mask shape: (88, 1024, 1024)
Building SAM predictor from checkpoint: checkpoints/MedSAM2_latest.pt


c:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\sam2\modeling\sam\transformer.py:23: UserWarning: Flash Attention is disabled as it requires a GPU with Ampere (8.0) CUDA capability.
  OLD_GPU, USE_FLASH_ATTN, MATH_KERNEL_ON = get_sdpa_settings()


iter=00 | thr=0.194962 | band=0.94 mm | error=2.06
iter=01 | thr=0.097481 | band=1.41 mm | error=1.59
iter=02 | thr=0.048740 | band=1.99 mm | error=1.01
iter=03 | thr=0.024370 | band=2.46 mm | error=0.54
iter=04 | thr=0.012185 | band=2.93 mm | error=0.07
iter=05 | thr=0.006093 | band=3.63 mm | error=0.63
iter=06 | thr=0.009139 | band=3.28 mm | error=0.28
iter=07 | thr=0.010662 | band=3.16 mm | error=0.16
iter=08 | thr=0.011424 | band=3.05 mm | error=0.05
iter=09 | thr=0.011804 | band=3.05 mm | error=0.05
iter=10 | thr=0.011995 | band=3.05 mm | error=0.05
iter=11 | thr=0.012090 | band=2.93 mm | error=0.07
iter=12 | thr=0.012042 | band=2.93 mm | error=0.07
iter=13 | thr=0.012019 | band=2.99 mm | error=0.01
iter=14 | thr=0.012007 | band=2.99 mm | error=0.01
iter=15 | thr=0.012001 | band=2.99 mm | error=0.01
iter=16 | thr=0.011998 | band=3.05 mm | error=0.05
iter=17 | thr=0.011999 | band=2.99 mm | error=0.01
iter=18 | thr=0.011998 | band=2.99 mm | error=0.01
iter=19 | thr=0.011998 | band=3

In [2]:
def run_prompt_strategy(data, prompt_dict, name, propagation_style="central_start"):
    """
    Runs one prompt strategy in a fresh Segmentation object.
    This avoids prompt/memory contamination between strategies.
    """
    seg_handler = Segmentation(data=data)

    seg_handler.compile_prompt_sets(
        prompt_dict_list=[prompt_dict],
        prompt_set_names=[name],
        prompt_set_weights=[1.0],
    )

    seg_handler.run_segmentation_sets(
        propagation_style=propagation_style,
        weighting_strategy="average",
        threshold=0.0,
    )

    seg_handler.remove_distant_slices(tolerance_frames=0)

    return np.asarray(seg_handler.predicted_seg).astype(bool), np.asarray(seg_handler.predicted_logits)

def run_prompt_strategy_sets(data, prompt_dicts, weightlist, namelist, propagation_style="central_start"):
    """
    Runs one prompt strategy in a fresh Segmentation object.
    This avoids prompt/memory contamination between strategies.
    """
    seg_handler = Segmentation(data=data)

    seg_handler.compile_prompt_sets(
        prompt_dict_list=prompt_dicts,
        prompt_set_names=namelist,
        prompt_set_weights=weightlist,
    )

    seg_handler.run_segmentation_sets(
        propagation_style=propagation_style,
        weighting_strategy="custom",
        threshold=0.0,

    )

    seg_handler.remove_distant_slices(tolerance_frames=0)

    return np.asarray(seg_handler.predicted_seg).astype(bool), np.asarray(seg_handler.predicted_logits)



In [3]:
root = r"C:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\data\LUNDPROBE\ExtendedSamples\development"

data = DataLoader(
    parentfolder=root,
    subject_nr=0,
    volume_of_interest="CTVT",
    verbose=True,
)

unc_handler = UG_prompter(data=data)

method = "local_normals"

unc_handler.threshold_uncertainty_map(
    unc_threshold=None,
    target_mm=2.0,
    method="raycast",
    mode="median",
)

unc_handler.compute_band_thickness(method=method)

tmp_seg = Segmentation(data=data)
dense_prompt = tmp_seg.load_dense_prompt()

nietjes_thr2 = unc_handler.generate_prompts_nietjes(
    unc_band_thr_mm=2.0,
    interpix_dist=3,
    pixel_interval=5,
    angle_step=5,
    method=method,
)

nietjes_thr3 = unc_handler.generate_prompts_nietjes(
    unc_band_thr_mm=3.0,
    interpix_dist=3,
    pixel_interval=10,
    angle_step=5,
    method=method,
)

nietjes_thr4 = unc_handler.generate_prompts_nietjes(
    unc_band_thr_mm=4.0,
    interpix_dist=3,
    pixel_interval=10,
    angle_step=5,
    method=method,
)

for z in nietjes_thr4:
    for key, value in nietjes_thr4[z].items():
        if isinstance(value, np.ndarray):
            nietjes_thr4[z][key] = value.copy()

bbox_no_dense_pad0 = unc_handler.generate_prompts_boxes(
    band_threshold=0.0,
    pad=0,
)

bbox_dense_pad0 = combine_prompt_sets([
    dense_prompt,
    unc_handler.generate_prompts_boxes(
        band_threshold=0.0,
        pad=0,
    )
])

bbox_dense_pad10 = combine_prompt_sets([
    dense_prompt,
    unc_handler.generate_prompts_boxes(
        band_threshold=0.0,
        pad=10,
    )
])

bbox_dense_pad25 = combine_prompt_sets([
    dense_prompt,
    unc_handler.generate_prompts_boxes(
        band_threshold=0.0,
        pad=25,
    )
])

prompt_strategies = {

    # Top row
    "Nietjes only":
        combine_prompt_sets([nietjes_thr3]),

    "Dense + nietjes\n2 mm":
        combine_prompt_sets([dense_prompt, nietjes_thr2]),

    "Dense + nietjes\n3 mm":
        combine_prompt_sets([dense_prompt, nietjes_thr3]),

    "Dense + nietjes\n4 mm":
        combine_prompt_sets([dense_prompt, nietjes_thr4]),


    #Second row
    "Nietjes only + BBox pad=0":
        combine_prompt_sets([nietjes_thr3, bbox_no_dense_pad0]),

    "dense + nietjes\n 2 mm + BBox pad=0":
        combine_prompt_sets([dense_prompt, nietjes_thr2, bbox_no_dense_pad0]),

    "dense + nietjes\n 3 mm + BBox pad=0":
        combine_prompt_sets([dense_prompt, nietjes_thr3, bbox_no_dense_pad0]),

    "dense + nietjes\n 4 mm + BBox npad=0":
        combine_prompt_sets([dense_prompt, nietjes_thr4, bbox_no_dense_pad0]),

    #Bottom row
    "BBox\npad=0":
        bbox_no_dense_pad0,

    "Dense + BBox\npad=0":
        bbox_dense_pad0,

    "Dense + BBox\npad=5":
        bbox_dense_pad10,

    "Dense + BBox\npad=10":
        bbox_dense_pad25,
}

prompt_set_strategies = {
    "Full ensemble 0.5-0.5": {
        "prompt_dicts": [combine_prompt_sets([dense_prompt, nietjes_thr4]), combine_prompt_sets([dense_prompt, bbox_no_dense_pad0])],
        "weights": [0.5,0.5],
        "names": ["Nietjes","Bbox"],
    },

    "Full ensemble 0.2-0.8": {
        "prompt_dicts": [combine_prompt_sets([dense_prompt, nietjes_thr4]), combine_prompt_sets([dense_prompt, bbox_no_dense_pad0])],
        "weights": [0.2,0.8],
        "names": ["Nietjes","Bbox"],
    },

    "Full ensemble 0.8-0.2": {
        "prompt_dicts": [combine_prompt_sets([dense_prompt, nietjes_thr4]), combine_prompt_sets([dense_prompt, bbox_no_dense_pad0])],
        "weights": [0.8, 0.2],
        "names": ["Nietjes","Bbox"],
    },

    "Full ensemble 0.4-0.6": {
        "prompt_dicts": [combine_prompt_sets([dense_prompt, nietjes_thr4]), combine_prompt_sets([dense_prompt, bbox_no_dense_pad0])],
        "weights": [0.4, 0.6],
        "names": ["Nietjes","Bbox"],
    },
}

Loaded subject newAcq_050f229dc2bdb64c with volume of interest 'CTVT'
Image shape: (88, 1024, 1024), Mask shape: (88, 1024, 1024), Uncertainty map shape: (88, 1024, 1024), Ground truth shape: (88, 1024, 1024)
Image spacing (z, y, x): (2.5, 0.46880000829696655, 0.46880000829696655)
Initilialized UG_prompter for subject 0 with volume of interest 'CTVT'
Image shape: (88, 1024, 1024), Mask shape: (88, 1024, 1024), Uncertainty map shape: (88, 1024, 1024).
iter=00 | thr=0.194962 | band=0.94 mm | error=1.06
iter=01 | thr=0.097481 | band=1.41 mm | error=0.59
iter=02 | thr=0.048740 | band=1.99 mm | error=0.01
iter=03 | thr=0.024370 | band=2.46 mm | error=0.46
iter=04 | thr=0.036555 | band=2.23 mm | error=0.23
iter=05 | thr=0.042648 | band=1.99 mm | error=0.01
iter=06 | thr=0.039602 | band=2.11 mm | error=0.11
iter=07 | thr=0.041125 | band=1.99 mm | error=0.01
iter=08 | thr=0.040363 | band=2.11 mm | error=0.11
iter=09 | thr=0.040744 | band=2.05 mm | error=0.05
iter=10 | thr=0.040934 | band=1.99 

In [ ]:
seg_results = {}
log_results = {}

for name, prompt_dict in prompt_strategies.items():

    print(f"Running: {name}")

    seg_results[name], log_results[name] = run_prompt_strategy(
        data=data,
        prompt_dict=prompt_dict,
        name=name,
        propagation_style="central_start",
    )

for strategy_name, config in prompt_set_strategies.items():

    print(f"Running: {strategy_name}")

    pred_seg, pred_logits = run_prompt_strategy_sets(
        data=data,
        prompt_dicts=config["prompt_dicts"],
        weightlist=config["weights"],
        namelist=config["names"],
        propagation_style="central_start"
    )

    seg_results[strategy_name] = pred_seg
    log_results[strategy_name] = pred_logits

Running: Nietjes only
Initialized Segmentation for subject 0 with volume of interest 'CTVT'
Mask shape: (88, 1024, 1024)
Building SAM predictor from checkpoint: checkpoints/MedSAM2_latest.pt
Running segmentation for prompt set 'Nietjes only' with slices: [28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 39, 40, 41, 42, 43, 44]
Device: cuda
Volume shape (D,H,W): (88, 1024, 1024)
Using prompts from slices: [28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 39, 40, 41, 42, 43, 44]
Adding prompt(s) on slice 28


c:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\sam2\sam2_video_predictor_npz.py:965: UserWarning: cannot import name '_C' from 'sam2' (c:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\sam2\__init__.py)

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  pred_masks_gpu = fill_holes_in_mask_scores(


Adding prompt(s) on slice 29
Adding prompt(s) on slice 30
Adding prompt(s) on slice 31
Adding prompt(s) on slice 32
Adding prompt(s) on slice 33
Adding prompt(s) on slice 34
Adding prompt(s) on slice 35
Adding prompt(s) on slice 36
Adding prompt(s) on slice 37
Adding prompt(s) on slice 39
Adding prompt(s) on slice 40
Adding prompt(s) on slice 41
Adding prompt(s) on slice 42
Adding prompt(s) on slice 43
Adding prompt(s) on slice 44
Forward propagation (from middle slice 36)...


propagate in video: 100%|██████████| 52/52 [00:12<00:00,  4.04it/s]


Backward propagation (from middle slice 36)...


...

Running segmentation for prompt set 'Bbox' with slices: [27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44]
Device: cuda
Volume shape (D,H,W): (88, 1024, 1024)
Using prompts from slices: [27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44]
Adding prompt(s) on slice 27
Adding prompt(s) on slice 28
Adding prompt(s) on slice 29
Adding prompt(s) on slice 30
Adding prompt(s) on slice 31
Adding prompt(s) on slice 32
Adding prompt(s) on slice 33
Adding prompt(s) on slice 34
Adding prompt(s) on slice 35
Adding prompt(s) on slice 36
Adding prompt(s) on slice 37
Adding prompt(s) on slice 38
Adding prompt(s) on slice 39
Adding prompt(s) on slice 40
Adding prompt(s) on slice 41
Adding prompt(s) on slice 42
Adding prompt(s) on slice 43
Adding prompt(s) on slice 44
Forward propagation (from middle slice 36)...


propagate in video: 100%|██████████| 52/52 [00:13<00:00,  3.75it/s]


Backward propagation (from middle slice 36)...


propagate in video: 100%|██████████| 37/37 [00:09<00:00,  4.08it/s]


Kept slices 27 to 44. Removed predicted segmentation outside dense mask ±0 slices.


In [5]:
plot_prompt_strategies = {}

plot_prompt_strategies.update(prompt_strategies)

for strategy_name, config in prompt_set_strategies.items():
    plot_prompt_strategies[strategy_name] = combine_prompt_sets(config["prompt_dicts"])

In [6]:
print(seg_results)

{'Nietjes only': array([[[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]],

       [[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False]],

       [[False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        [False, False, False, ..., False, False, False],
        ...,
        [False, False, False, ..., False, False, False],
        [False, False, False

In [7]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from skimage.measure import find_contours


def interactive_prompt_strategy_comparison(
    data,
    seg_results,
    prompt_strategies,
    zoom_fraction=0.20,
    figsize_per_plot=4,
):
    img = np.asarray(data.img)
    dense = np.asarray(data.mask).astype(bool)

    n_slices = img.shape[0]
    strategy_names = list(seg_results.keys())

    def draw_contour(ax, mask, color, linewidth=1.0, linestyle="-"):
        if mask is None or not np.any(mask):
            return

        contours = find_contours(mask.astype(float), 0.5)

        for contour in contours:
            ax.plot(
                contour[:, 1],
                contour[:, 0],
                color=color,
                linewidth=linewidth,
                linestyle=linestyle,
            )

    def draw_prompts(ax, prompt_dict, z, x0, y0):
        if z not in prompt_dict:
            return

        prompts = prompt_dict[z]

        points = prompts.get("points", None)
        labels = prompts.get("point_labels", None)

        if points is not None and labels is not None:
            points = np.asarray(points)
            labels = np.asarray(labels)

            pos = points[labels == 1]
            neg = points[labels == 0]

            if len(pos) > 0:
                ax.scatter(
                    pos[:, 0] - x0,
                    pos[:, 1] - y0,
                    marker="+",
                    s=25,
                    linewidths=1.5,
                    color="lime",
                    label="Positive prompt",
                )

            if len(neg) > 0:
                ax.scatter(
                    neg[:, 0] - x0,
                    neg[:, 1] - y0,
                    marker="x",
                    s=25,
                    linewidths=1.3,
                    color="red",
                    label="Negative prompt",
                )

        boxes = prompts.get("boxes", None)

        if boxes is None:
            boxes = prompts.get("bbox", None)

        if boxes is not None:
            boxes = np.asarray(boxes)

            if boxes.ndim == 1:
                boxes = boxes[None, :]

            for box in boxes:
                x_min, y_min, x_max, y_max = box

                rect = plt.Rectangle(
                    (x_min - x0, y_min - y0),
                    x_max - x_min,
                    y_max - y_min,
                    fill=False,
                    linestyle="--",
                    edgecolor="blue",
                    linewidth=1,
                )

                ax.add_patch(rect)

    def get_crop(z):
        base_mask = dense[z]

        if np.any(base_mask):
            ys, xs = np.where(base_mask)
            cy, cx = ys.mean(), xs.mean()
        else:
            cy, cx = np.array(img.shape[1:]) / 2

        H, W = img.shape[1:]
        crop_h = int(H * zoom_fraction)
        crop_w = int(W * zoom_fraction)

        y0 = max(0, int(cy - crop_h // 2))
        y1 = min(H, int(cy + crop_h // 2))
        x0 = max(0, int(cx - crop_w // 2))
        x1 = min(W, int(cx + crop_w // 2))

        return x0, x1, y0, y1

    valid_slices = np.where(np.any(dense, axis=(1, 2)))[0]

    if len(valid_slices) > 0:
        default_slice = int(valid_slices[len(valid_slices) // 2])
    else:
        default_slice = n_slices // 2

    slice_slider = widgets.IntSlider(
        value=default_slice,
        min=0,
        max=n_slices - 1,
        step=1,
        description="Slice:",
        continuous_update=False,
    )

    output = widgets.Output()

    def update_plot(change=None):
        z = slice_slider.value

        with output:
            clear_output(wait=True)

            x0, x1, y0, y1 = get_crop(z)

            n = len(strategy_names)

            ncols = 4
            nrows = int(np.ceil(n / ncols))

            fig, axes = plt.subplots(
                nrows,
                ncols,
                figsize=(figsize_per_plot * ncols,
                        figsize_per_plot * nrows),
                squeeze=False,
            )

            axes_flat = axes.flatten()

            for ax, name in zip(axes_flat, strategy_names):
                pred = np.asarray(seg_results[name]).astype(bool)
                prompt_dict = prompt_strategies[name]

                ax.imshow(img[z, y0:y1, x0:x1], cmap="gray")

                # Dense mask contour: red
                draw_contour(
                    ax,
                    dense[z, y0:y1, x0:x1],
                    color="orange",
                    linewidth=0.8,
                )

                # Updated segmentation contour: yellow
                draw_contour(
                    ax,
                    pred[z, y0:y1, x0:x1],
                    color="yellow",
                    linewidth=1.0,
                )

                draw_prompts(
                    ax=ax,
                    prompt_dict=prompt_dict,
                    z=z,
                    x0=x0,
                    y0=y0,
                )

                ax.set_title(name)
                ax.axis("off")

            fig.suptitle(
                f"Prompt strategy comparison | slice {z}",
                fontsize=14,
            )

            plt.tight_layout()
            plt.show()

    slice_slider.observe(update_plot, names="value")

    display(slice_slider, output)
    update_plot()

In [8]:
interactive_prompt_strategy_comparison(
    data=data,
    seg_results=seg_results,
    prompt_strategies=plot_prompt_strategies,
    zoom_fraction=0.15,
)

IntSlider(value=36, continuous_update=False, description='Slice:', max=87)

Output()

In [9]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from skimage.measure import find_contours


def save_prompt_strategy_comparison_pngs(
    data,
    seg_results,
    prompt_strategies,
    output_folder="Prompt_Strategy_Comparison",
    zoom_fraction=0.35,
    figsize_per_plot=4,
    dpi=300,
):
    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    img = np.asarray(data.img)
    dense = np.asarray(data.mask).astype(bool)
    gt = np.asarray(data.gt).astype(bool)

    strategy_names = list(seg_results.keys())

    def draw_contour(ax, mask, color, linewidth=1.0, linestyle="-"):
        if mask is None or not np.any(mask):
            return

        contours = find_contours(mask.astype(float), 0.5)

        for contour in contours:
            ax.plot(
                contour[:, 1],
                contour[:, 0],
                color=color,
                linewidth=linewidth,
                linestyle=linestyle,
            )

    def draw_prompts(ax, prompt_dict, z, x0, y0):
        if z not in prompt_dict:
            return

        prompts = prompt_dict[z]

        points = prompts.get("points", None)
        labels = prompts.get("point_labels", None)

        if points is not None and labels is not None:
            points = np.asarray(points)
            labels = np.asarray(labels)

            pos = points[labels == 1]
            neg = points[labels == 0]

            if len(pos) > 0:
                ax.scatter(
                    pos[:, 0] - x0,
                    pos[:, 1] - y0,
                    marker="+",
                    s=25,
                    linewidths=1.5,
                    color="lime",
                )

            if len(neg) > 0:
                ax.scatter(
                    neg[:, 0] - x0,
                    neg[:, 1] - y0,
                    marker="x",
                    s=25,
                    linewidths=1.3,
                    color="red",
                )

        boxes = prompts.get("boxes", None)

        if boxes is None:
            boxes = prompts.get("bbox", None)

        if boxes is not None:
            boxes = np.asarray(boxes)

            if boxes.ndim == 1:
                boxes = boxes[None, :]

            for box in boxes:
                x_min, y_min, x_max, y_max = box

                rect = plt.Rectangle(
                    (x_min - x0, y_min - y0),
                    x_max - x_min,
                    y_max - y_min,
                    fill=False,
                    linestyle="--",
                    edgecolor="cyan",
                    linewidth=1,
                )

                ax.add_patch(rect)

    def get_crop(z):
        base_mask = dense[z]

        if np.any(base_mask):
            ys, xs = np.where(base_mask)
            cy, cx = ys.mean(), xs.mean()
        else:
            cy, cx = np.array(img.shape[1:]) / 2

        H, W = img.shape[1:]

        crop_h = int(H * zoom_fraction)
        crop_w = int(W * zoom_fraction)

        y0 = max(0, int(cy - crop_h // 2))
        y1 = min(H, int(cy + crop_h // 2))
        x0 = max(0, int(cx - crop_w // 2))
        x1 = min(W, int(cx + crop_w // 2))

        return x0, x1, y0, y1

    slices_to_save = np.where(np.any(gt, axis=(1, 2)))[0].tolist()

    print(f"Saving {len(slices_to_save)} GT-containing slices:")
    print(slices_to_save)

    for z in slices_to_save:
        x0, x1, y0, y1 = get_crop(z)

        n = len(strategy_names)
        ncols = 4
        nrows = int(np.ceil(n / ncols))

        fig, axes = plt.subplots(
            nrows,
            ncols,
            figsize=(figsize_per_plot * ncols, figsize_per_plot * nrows),
            squeeze=False,
        )

        axes_flat = axes.flatten()

        for ax, name in zip(axes_flat, strategy_names):
            pred = np.asarray(seg_results[name]).astype(bool)
            prompt_dict = prompt_strategies[name]

            ax.imshow(img[z, y0:y1, x0:x1], cmap="gray")

            if (
                z in prompt_dict
                and prompt_dict[z].get("mask_input", None) is not None
            ):
                draw_contour(
                    ax,
                    dense[z, y0:y1, x0:x1],
                    color="orange",
                    linewidth=0.8,
                )

            draw_contour(
                ax,
                pred[z, y0:y1, x0:x1],
                color="yellow",
                linewidth=1.0,
            )

            draw_prompts(
                ax=ax,
                prompt_dict=prompt_dict,
                z=z,
                x0=x0,
                y0=y0,
            )

            ax.set_title(name)
            ax.axis("off")

        for ax in axes_flat[len(strategy_names):]:
            ax.axis("off")

        fig.suptitle(
            f"Prompt strategy comparison | slice {z}",
            fontsize=14,
        )

        plt.tight_layout()

        save_path = output_folder / f"slice_{z:03d}.png"

        fig.savefig(
            save_path,
            dpi=dpi,
            bbox_inches="tight",
        )

        plt.close(fig)

        print(f"Saved {save_path}")

    print("Done!")

In [10]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from skimage.measure import find_contours


def interactive_logit_comparison(
    data,
    logit_results,
    seg_results=None,
    zoom_fraction=0.35,
    figsize_per_plot=4,
    alpha=0.55,
    cmap="magma",
    vmin=None,
    vmax=None,
):

    img = np.asarray(data.img)
    dense = np.asarray(data.mask).astype(bool)

    strategy_names = list(logit_results.keys())
    n_slices = img.shape[0]

    if vmin is None:
        vmin = min(np.nanpercentile(np.asarray(logit_results[name]), 1) for name in strategy_names)

    if vmax is None:
        vmax = max(np.nanpercentile(np.asarray(logit_results[name]), 99) for name in strategy_names)

    def draw_contour(ax, mask, color="lime", linewidth=1.1):
        if mask is None or not np.any(mask):
            return

        contours = find_contours(mask.astype(float), 0.5)

        for contour in contours:
            ax.plot(
                contour[:, 1],
                contour[:, 0],
                color=color,
                linewidth=linewidth,
            )

    def get_crop(z):

        if np.any(dense[z]):
            ys, xs = np.where(dense[z])
            cy, cx = ys.mean(), xs.mean()
        else:
            cy, cx = np.array(img.shape[1:]) / 2

        H, W = img.shape[1:]

        crop_h = int(H * zoom_fraction)
        crop_w = int(W * zoom_fraction)

        y0 = max(0, int(cy - crop_h // 2))
        y1 = min(H, int(cy + crop_h // 2))
        x0 = max(0, int(cx - crop_w // 2))
        x1 = min(W, int(cx + crop_w // 2))

        return x0, x1, y0, y1

    valid_slices = np.where(np.any(dense, axis=(1, 2)))[0]

    if len(valid_slices):
        default_slice = int(valid_slices[len(valid_slices) // 2])
    else:
        default_slice = n_slices // 2

    slider = widgets.IntSlider(
        value=default_slice,
        min=0,
        max=n_slices - 1,
        description="Slice:",
        continuous_update=False,
    )

    def plot(z):

        x0, x1, y0, y1 = get_crop(z)

        n = len(strategy_names)
        ncols = 4
        nrows = int(np.ceil(n / ncols))

        fig, axes = plt.subplots(
            nrows,
            ncols,
            figsize=(figsize_per_plot * ncols,
                     figsize_per_plot * nrows),
            squeeze=False,
        )

        axes = axes.flatten()

        for ax, name in zip(axes, strategy_names):

            logits = np.asarray(logit_results[name])

            ax.imshow(
                img[z, y0:y1, x0:x1],
                cmap="gray",
            )

            im = ax.imshow(
                logits[z, y0:y1, x0:x1],
                cmap=cmap,
                alpha=alpha,
                vmin=vmin,
                vmax=vmax,
            )

            if seg_results is not None and name in seg_results:
                pred = np.asarray(seg_results[name]).astype(bool)
                draw_contour(
                    ax,
                    pred[z, y0:y1, x0:x1],
                    color="lime",
                    linewidth=1.1,
                )

            ax.set_title(name)
            ax.axis("off")

        for ax in axes[len(strategy_names):]:
            ax.axis("off")

        fig.subplots_adjust(
            right=0.90,
            wspace=0.05,
            hspace=0.20,
        )

        cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.70])
        fig.colorbar(im, cax=cbar_ax, label="Logit")

        plt.show()

    widgets.interact(plot, z=slider)

In [11]:
interactive_logit_comparison(
    data=data,
    logit_results=log_results,
    seg_results=seg_results,
    cmap="turbo",
    alpha=0.50,
    vmin=-10,
    vmax=10,
)

interactive(children=(IntSlider(value=36, continuous_update=False, description='Slice:', max=87), Output()), _…

In [12]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from skimage.measure import find_contours


def save_logit_comparison_pngs(
    data,
    logit_results,
    seg_results=None,
    output_folder="Logit_Comparison",
    zoom_fraction=0.35,
    figsize_per_plot=4,
    alpha=0.55,
    cmap="magma",
    vmin=-10,
    vmax=10,
    dpi=300,
):

    output_folder = Path(output_folder)
    output_folder.mkdir(parents=True, exist_ok=True)

    img = np.asarray(data.img)
    dense = np.asarray(data.mask).astype(bool)
    gt = np.asarray(data.gt).astype(bool)

    strategy_names = list(logit_results.keys())

    def draw_contour(ax, mask, color="lime", linewidth=1.1):
        if mask is None or not np.any(mask):
            return

        contours = find_contours(mask.astype(float), 0.5)

        for contour in contours:
            ax.plot(
                contour[:, 1],
                contour[:, 0],
                color=color,
                linewidth=linewidth,
            )

    def get_crop(z):

        if np.any(gt[z]):
            ys, xs = np.where(gt[z])
            cy, cx = ys.mean(), xs.mean()
        elif np.any(dense[z]):
            ys, xs = np.where(dense[z])
            cy, cx = ys.mean(), xs.mean()
        else:
            cy, cx = np.array(img.shape[1:]) / 2

        H, W = img.shape[1:]

        crop_h = int(H * zoom_fraction)
        crop_w = int(W * zoom_fraction)

        y0 = max(0, int(cy - crop_h // 2))
        y1 = min(H, int(cy + crop_h // 2))
        x0 = max(0, int(cx - crop_w // 2))
        x1 = min(W, int(cx + crop_w // 2))

        return x0, x1, y0, y1

    gt_slices = np.where(np.any(gt, axis=(1, 2)))[0]

    for z in gt_slices:

        x0, x1, y0, y1 = get_crop(z)

        n = len(strategy_names)
        ncols = 4
        nrows = int(np.ceil(n / ncols))

        fig, axes = plt.subplots(
            nrows,
            ncols,
            figsize=(figsize_per_plot * ncols,
                     figsize_per_plot * nrows),
            squeeze=False,
        )

        axes = axes.flatten()

        for ax, name in zip(axes, strategy_names):

            logits = np.asarray(logit_results[name])

            ax.imshow(
                img[z, y0:y1, x0:x1],
                cmap="gray",
            )

            im = ax.imshow(
                logits[z, y0:y1, x0:x1],
                cmap=cmap,
                alpha=alpha,
                vmin=vmin,
                vmax=vmax,
            )

            if seg_results is not None and name in seg_results:
                pred = np.asarray(seg_results[name]).astype(bool)
                draw_contour(
                    ax,
                    pred[z, y0:y1, x0:x1],
                    color="lime",
                    linewidth=1.1,
                )

            ax.set_title(name)
            ax.axis("off")

        for ax in axes[len(strategy_names):]:
            ax.axis("off")

        fig.subplots_adjust(
            right=0.90,
            wspace=0.05,
            hspace=0.20,
        )

        cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.70])
        fig.colorbar(im, cax=cbar_ax, label="Logit")

        fig.suptitle(
            f"Logit comparison | GT slice {z}",
            fontsize=14,
        )

        save_path = output_folder / f"logit_comparison_slice_{z:03d}.png"

        fig.savefig(
            save_path,
            dpi=dpi,
            bbox_inches="tight",
        )

        plt.close(fig)

    print(f"Saved {len(gt_slices)} figures to: {output_folder}")

In [13]:
save_prompt_strategy_comparison_pngs(
    data=data,
    seg_results=seg_results,
    prompt_strategies=plot_prompt_strategies,
    output_folder="unc_2.0mm_Prompt_Strategy_Comparison_subject_1",
    zoom_fraction=0.15,
    figsize_per_plot=4,
    dpi=300,
)

save_logit_comparison_pngs(
    data=data,
    logit_results=log_results,
    seg_results=seg_results,
    output_folder="unc_2.0mm_Prompt_Strategy_Comparison_subject_1",
    cmap="turbo",
    alpha=0.50,
    vmin=-10,
    vmax=10,
)


Saving 16 GT-containing slices:
[29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44]
Saved unc_2.0mm_Prompt_Strategy_Comparison_subject_1\slice_029.png
Saved unc_2.0mm_Prompt_Strategy_Comparison_subject_1\slice_030.png
Saved unc_2.0mm_Prompt_Strategy_Comparison_subject_1\slice_031.png
Saved unc_2.0mm_Prompt_Strategy_Comparison_subject_1\slice_032.png
Saved unc_2.0mm_Prompt_Strategy_Comparison_subject_1\slice_033.png
Saved unc_2.0mm_Prompt_Strategy_Comparison_subject_1\slice_034.png
Saved unc_2.0mm_Prompt_Strategy_Comparison_subject_1\slice_035.png
Saved unc_2.0mm_Prompt_Strategy_Comparison_subject_1\slice_036.png
Saved unc_2.0mm_Prompt_Strategy_Comparison_subject_1\slice_037.png
Saved unc_2.0mm_Prompt_Strategy_Comparison_subject_1\slice_038.png
Saved unc_2.0mm_Prompt_Strategy_Comparison_subject_1\slice_039.png
Saved unc_2.0mm_Prompt_Strategy_Comparison_subject_1\slice_040.png
Saved unc_2.0mm_Prompt_Strategy_Comparison_subject_1\slice_041.png
Saved unc_2.0mm_Prompt_Strategy_